# 03 - Rio 2016: medals vs delegation size (interactive)

**Question.** How strongly does a country's medal total depend on the size of its delegation,
and which countries were the most efficient?

**Data.** `data/olympia2016_athletes.csv` - 11,538 athletes with nationality, sport and medal
counts (Rio 2016, public dataset distributed with the CISC7204 tutorial).


In [1]:
import numpy as np
import pandas as pd
from bokeh.io import output_notebook
from bokeh.models import ColumnDataSource, HoverTool
from bokeh.plotting import figure, show
from ipywidgets import interact, IntSlider

output_notebook()
dataset = pd.read_csv('../data/olympia2016_athletes.csv')
print("athletes:", dataset.shape[0], "| countries:", dataset['nationality'].nunique())
dataset.head()


Loading BokehJS ...

athletes: 11538 | countries: 207


,id,name,nationality,sex,dob,height,weight,sport,gold,silver,bronze
0,736041664,A Jesus Garcia,ESP,male,10/17/69,1.72,64.0,athletics,0,0,0
1,532037425,A Lam Shin,KOR,female,9/23/86,1.68,56.0,fencing,0,0,0
2,435962603,Aaron Brown,CAN,male,5/27/92,1.98,79.0,athletics,0,0,1
3,521041435,Aaron Cook,MDA,male,1/2/91,1.83,80.0,taekwondo,0,0,0
4,33922579,Aaron Gate,NZL,male,11/26/90,1.81,71.0,cycling,0,0,0


## Aggregate to one row per country


In [2]:
countries = (dataset.groupby('nationality')
             .agg(athletes=('id', 'count'),
                  gold=('gold', 'sum'),
                  silver=('silver', 'sum'),
                  bronze=('bronze', 'sum'))
             .reset_index())
countries['medals'] = countries['gold'] + countries['silver'] + countries['bronze']
countries['medals_per_athlete'] = countries['medals'] / countries['athletes']
r = np.corrcoef(countries['athletes'], countries['medals'])[0, 1]
print("countries:", len(countries), "| Pearson r(athletes, medals) = %.3f" % r)
countries.sort_values('medals', ascending=False).head(5)


countries: 207 | Pearson r(athletes, medals) = 0.869


,nationality,athletes,gold,silver,bronze,medals,medals_per_athlete
198,USA,567,139,54,71,264,0.465608
72,GER,441,49,44,67,160,0.362812
68,GBR,374,64,55,26,145,0.387701
159,RUS,286,52,29,34,115,0.402098
38,CHN,404,46,30,37,113,0.279703


## The interactive scatter

Two sliders filter the countries; a `HoverTool` shows the medal split on hover.


In [3]:
max_athletes = int(countries['athletes'].max())
max_medals = int(countries['medals'].max())

def get_plot(max_athletes, max_medals):
    sub = countries[(countries['athletes'] <= max_athletes) &
                    (countries['medals'] <= max_medals)]
    source = ColumnDataSource(sub)
    plot = figure(title='Rio Olympics 2016 - Medal comparison',
                  x_axis_label='Number of Medals', y_axis_label='Num of Athletes',
                  width=820, height=500)
    plot.scatter('medals', 'athletes', source=source, size=9, marker='circle',
                 fill_color='#2678b2', line_color='white', alpha=0.85)
    plot.add_tools(HoverTool(tooltips=[('Country', '@nationality'),
                                       ('Athletes', '@athletes'),
                                       ('Medals', '@medals'),
                                       ('Gold', '@gold'), ('Silver', '@silver'),
                                       ('Bronze', '@bronze')]))
    print("countries shown:", len(sub))
    return plot

athletes_slider = IntSlider(min=0, max=max_athletes, value=max_athletes,
                            description='Max Athletes', orientation='vertical')
medals_slider = IntSlider(min=0, max=max_medals, value=max_medals,
                          description='Max Medals', orientation='horizontal')

@interact(max_athletes=athletes_slider, max_medals=medals_slider)
def show_plot(max_athletes, max_medals):
    show(get_plot(max_athletes, max_medals))


interactive(children=(IntSlider(value=567, description='Max Athletes', max=567, orientation='vertical'), IntSl…

## The efficiency ranking (countries with at least 20 athletes)


In [4]:
big = countries[countries['athletes'] >= 20].copy()
big['per_athlete'] = big['medals'] / big['athletes']
print(big.sort_values('per_athlete', ascending=False)
          .head(5)[['nationality', 'athletes', 'medals', 'per_athlete']]
          .to_string(index=False))
print()
print(big.sort_values('per_athlete').head(3)[['nationality', 'athletes', 'medals', 'per_athlete']]
          .to_string(index=False))


nationality  athletes  medals  per_athlete
        JAM        57      30     0.526316
        SRB       103      53     0.514563
        USA       567     264     0.465608
        RUS       286     115     0.402098
        GBR       374     145     0.387701



nationality  athletes  medals  per_athlete
        ZIM        35       0          0.0
        GUA        21       0          0.0
        HKG        38       0          0.0


**Result.** Medal totals are strongly associated with delegation size (Pearson **r = 0.87**
across 207 countries): the USA sent 567 athletes and won 264 medals. Among countries with at
least 20 athletes, Jamaica (57 athletes -> 30 medals, 0.53 per athlete) and Serbia (103 -> 53,
0.51) converted athletes into medals more efficiently than the USA (0.47).

**Limitation.** This is a descriptive association, not a causal one: bigger delegations compete
in more events, and the data counts athletes rather than medal events, so team sports multiply
one medal across many athletes. The efficiency ratio therefore penalises team-sport nations.
